# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Exploring Hypothesis 1

---
## Foreword

In this notebook, my goal is to explore this hypothesis:

$$
\mathbb{E}[\Delta\textit{users\_close}_{i, t + 1} \mid \text{high}(ASVI_{i, t})] \ge 0
$$

where $ASVI_{i, t}$ translates abnormal volume in Google Trends. In other words, the purpose of this hypothesis is to explore whether lagged attention spikes in Google Trends can predict an increase in the number of Robinhood users holding a stock $i$ at time $t+1$. Additionally, I will also focus on the following question: "Among competing attention proxies, which ones contain incremental information about next-day retail demand on Robinhood?"


## 1. Libraries & Data

I first load the relevant libraries and data.

In [2]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from linearmodels import OLS

In [3]:
# data
main = pd.read_csv("../../../data/processed/main_with_svi.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_3556/1285260102.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/main_with_svi.csv")


Once I loaded the data, I parse the date accordingly.

In [4]:
# date format
main["date"] = pd.to_datetime(main["date"])

## 2. Exploratory Data Analysis

In this section, I conduct a brief EDA.

In [5]:
# the columns
main.columns

Index(['date', 'users_close', 'users_last', 'ticker', 'permno', 'ret', 'prc',
       'vol', 'shrout', 'exchcd', 'symro', 'symsu', 'buy_num_trades_LR',
       'sell_num_trades_LR', 'total_trade_LR', 'buy_vol_LR', 'sell_vol_LR',
       'total_vol_LR', 'close_price', 'open_price', 'close_vol', 'open_vol',
       'total_vol_m', 'intra_ret', 'buy_num_trades_tick',
       'sell_num_trades_tick', 'buy_vol_tick', 'sell_vol_tick',
       'total_trade_tick', 'buy_num_trades_wrds', 'sell_num_trades_wrds',
       'buy_vol_wrds', 'sell_vol_wrds', 'total_trade_wrds', 'bs_ratio_num',
       'bs_ratio_vol', 'buy_num_trades_retail', 'sell_num_trades_retail',
       'buy_vol_retail', 'sell_vol_retail', 'total_trade_retail',
       'total_vol_retail', 'bs_ratio_retail_vol', 'bs_ratio_retail_num',
       'intra_volatility', 'buy_num_trades_inst50k', 'sell_num_trades_inst50k',
       'buy_vol_inst50k', 'sell_vol_inst50k', 'total_trade_inst50k',
       'total_vol_inst50k', 'bs_ratio_inst50k_vol', 'bs_ratio_

In [6]:
# rename TSSVI to svi
main.rename(columns={"TSSVI": "svi"}, inplace=True)

In [7]:
# number of non-nan rows with svi and number of unique tickers in those rows
print(f"Number of non-nan rows with svi: {main['svi'].notna().sum()}")
print(f"Number of unique tickers in those rows: {main[main['date'].notna() & main['svi'].notna()]['ticker'].nunique()}")

Number of non-nan rows with svi: 871692
Number of unique tickers in those rows: 1698


In [ ]:
# shape, number of unique tickers
print(f"Shape of the dataset: {main.shape}")
print(f"Number of unique tickers: {main['ticker'].nunique()}")

Shape of the dataset: (4431180, 59)
Number of unique tickers: 9007


In [15]:
# number of unique tickers with non-nan ess, css, and anl_chg
print(f"Number of unique tickers with non-nan ess: {main[main['ess'].notna()]['ticker'].nunique()}")
print(f"Number of unique tickers with non-nan css: {main[main['css'].notna()]['ticker'].nunique()}")
print(f"Number of unique tickers with non-nan anl_chg: {main[main['anl_chg'].notna()]['ticker'].nunique()}")

Number of unique tickers with non-nan ess: 4413
Number of unique tickers with non-nan css: 4483
Number of unique tickers with non-nan anl_chg: 4483


## 3. Regressions

### 3.1. Basline Regression (Abnormal Volume and Returns)

The first regression is specifed as follows.

**Dependent Variable**

$$
\Delta \textit{users\_close}_{i, t} = {\textit{users\_close}_{i, t} - \textit{users\_close}_{i, t -1}}
$$

**Independent Variables**

1. abnormal returns

$$
\textit{ab\_ret}_{i, t - 1} = \ln(\frac{ret_{i, t - 1}}{\frac{1}{20}\sum_{k = 2}^{21} ret_{i, t - k}})
$$

2. abnormal volume

$$
\textit{ab\_vol}_{i, t} = \ln(\frac{vol_{i, t - 1}}{\frac{1}{20}\sum_{k = 2}^{21} vol_{i, t - k}})
$$

In [9]:
# select only relevant variables
df_reg_1 = main[['date', 'ticker', 'users_close', 'ret', 'vol', 'svi', 'num_news', 'ess', 'css', 'anl_chg']].copy()

# sort by ticker and date
df_reg_1.sort_values(by=['ticker', 'date'], inplace=True)

# drop rows with missing raw inputs
df_reg_1.dropna(subset=['users_close', 'ret', 'vol'], inplace=True)

# userchg
df_reg_1['userchg'] = df_reg_1.groupby('ticker')['users_close'].diff()

# userchg(t - 1)
df_reg_1['userchg_lag1'] = df_reg_1.groupby('ticker')['userchg'].shift(1)

# ln userchg
df_reg_1['ln_userchg'] = np.log(1 + np.abs(df_reg_1['userchg']))

# ab_ret_lag1
def make_ab_ret(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    sigma = lags.std(axis=1)
    z = (s.shift(1) - mu) / sigma
    z[sigma == 0] = np.nan
    return z

df_reg_1["ab_ret_lag1"] = df_reg_1.groupby("ticker")["ret"].transform(make_ab_ret)

# ret(t - 1)
df_reg_1['ret_lag1'] = df_reg_1.groupby('ticker')['ret'].shift(1)

# ab_vol_lag1
def make_ab_vol(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_vol_lag1"] = df_reg_1.groupby("ticker")["vol"].transform(make_ab_vol)

# abnormal svi: ln(svi(t - 1) / mean(svi(t - 2), ..., svi(t - 21)))
def make_ab_svi(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_svi_lag1"] = df_reg_1.groupby("ticker")["svi"].transform(make_ab_svi)

# abnormal news: ln(num_news(t - 1) / mean(num_news(t - 2), ..., num_news(t - 21)))
def make_ab_news(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

df_reg_1["ab_news_lag1"] = df_reg_1.groupby("ticker")["num_news"].transform(make_ab_news)

# lagged ess, css, and anl_chg
df_reg_1['ess_lag1'] = df_reg_1.groupby('ticker')['ess'].shift(1)
df_reg_1['css_lag1'] = df_reg_1.groupby('ticker')['css'].shift(1)
df_reg_1['anl_chg_lag1'] = df_reg_1.groupby('ticker')['anl_chg'].shift(1)

# optional cleanup
df_reg_1.replace([np.inf, -np.inf], np.nan, inplace=True)

In [10]:
reg_data = df_reg_1.dropna().copy()

# make sure date is datetime
reg_data['date'] = pd.to_datetime(reg_data['date'])

# convert clusters to integer codes
reg_data['ticker_clust'] = reg_data['ticker'].astype('category').cat.codes
reg_data['date_clust'] = reg_data['date'].astype('category').cat.codes

X1 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1']])
y1 = reg_data['userchg']

results1 = OLS(y1, X1).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results1.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2354
Estimator:                        OLS   Adj. R-squared:                 0.2354
No. Observations:              180751   F-statistic:                    159.00
Date:                Mon, Mar 23 2026   P-value (F-stat)                0.0000
Time:                        11:49:07   Distribution:                  chi2(3)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            2.6020     3.0806     0.8446     0.3983     -3.4360      8.6399
ret_lag1         253.88     124.53     2.038

In [11]:
X2 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1', 'ab_svi_lag1']])
y2 = reg_data['userchg']

results2 = OLS(y2, X2).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results2.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2355
Estimator:                        OLS   Adj. R-squared:                 0.2354
No. Observations:              180751   F-statistic:                    206.66
Date:                Mon, Mar 23 2026   P-value (F-stat)                0.0000
Time:                        11:49:21   Distribution:                  chi2(4)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            1.8057     2.9205     0.6183     0.5364     -3.9183      7.5298
ret_lag1         253.71     124.54     2.037

In [12]:
X3 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1', 'ab_svi_lag1', 'ab_news_lag1']])
y3 = reg_data['userchg']

results3 = OLS(y3, X3).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results3.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2355
Estimator:                        OLS   Adj. R-squared:                 0.2354
No. Observations:              180751   F-statistic:                    259.03
Date:                Mon, Mar 23 2026   P-value (F-stat)                0.0000
Time:                        11:49:42   Distribution:                  chi2(5)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            2.8833     2.6775     1.0769     0.2815     -2.3644      8.1311
ret_lag1         253.50     124.39     2.038

In [13]:
X4 = sm.add_constant(reg_data[['ret_lag1', 'ab_vol_lag1', 'userchg_lag1',
                               'ab_svi_lag1', 'ab_news_lag1', 'ess_lag1', 'css_lag1', 'anl_chg_lag1']])
y4 = reg_data['userchg']

results4 = OLS(y4, X4).fit(
    cov_type='clustered',
    clusters=reg_data[['ticker_clust', 'date_clust']]
)

print(results4.summary)

                            OLS Estimation Summary                            
Dep. Variable:                userchg   R-squared:                      0.2355
Estimator:                        OLS   Adj. R-squared:                 0.2355
No. Observations:              180751   F-statistic:                    297.12
Date:                Mon, Mar 23 2026   P-value (F-stat)                0.0000
Time:                        11:50:06   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            3.6706     2.6400     1.3904     0.1644     -1.5036      8.8448
ret_lag1         256.08     124.74     2.053